# Phase 3 - Notebook 07: Running DUSt3R on Custom Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase3/07_custom_data.ipynb)


## Setup


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from src.dust3r.pointmap import PointMap
from src.dust3r.alignment import GlobalAligner
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print("Setup complete!")

## 1. 数据准备指南

### DUSt3R 的输入要求

1. **图像格式**
   - 格式: JPG, PNG
   - 分辨率: 256×256 至 1024×1024
   - 无需标定相机参数！

2. **场景覆盖**
   - 至少 2 张图像（图像对）
   - 推荐 5-20 张用于完整重建
   - 视角基线：20°-45° 角度差

3. **光照条件**
   - 避免极端逆光/顺光
   - 一致的光照最佳


## 2. 文件组织


In [ ]:
# 推荐的目录结构
print("推荐的目录结构：")
print()
print("my_scene/")
print("├── images/")
print("│   ├── 001.jpg")
print("│   ├── 002.jpg")
print("│   ├── 003.jpg")
print("│   └── ...")
print("├── cameras.txt  (可选: COLMAP 格式，如果有的话)")
print("└── metadata.json (可选: 拍摄参数)")
print()
print("DUSt3R 不需要 cameras.txt！")

## 3. 批量处理图像对


In [ ]:
def process_image_pairs(image_dir, num_images=5):
    """
    生成图像对并处理。
    """
    import os
    
    # 获取图像列表
    image_files = sorted([f for f in os.listdir(image_dir) 
                         if f.lower().endswith(('.jpg', '.png'))])
    
    # 生成所有配对
    pairs = []
    for i in range(len(image_files)):
        for j in range(i+1, len(image_files)):
            pairs.append((image_files[i], image_files[j]))
    
    print(f"找到 {len(image_files)} 张图像")
    print(f"生成 {len(pairs)} 个图像对")
    
    return pairs[:num_images*2]  # 限制处理数量

# 演示
pairs = [('001.jpg', '002.jpg'), ('002.jpg', '003.jpg'), ('001.jpg', '003.jpg')]
print(f"\n示例图像对: {pairs}")

## 4. 完整的 DUSt3R 推理脚本


In [ ]:
def run_dust3r_on_scene(image_dir, output_dir=None, conf_threshold=0.5):
    """
    对整个场景运行 DUSt3R。
    
    Args:
        image_dir: 包含图像的目录
        output_dir: 输出目录
        conf_threshold: 置信度阈值
    
    Returns:
        aligner: GlobalAligner 对象（包含全局点云）
    """
    import os
    
    # 获取图像列表
    image_files = sorted([f for f in os.listdir(image_dir)
                         if f.lower().endswith(('.jpg', '.png'))])
    
    # 初始化对齐器
    aligner = GlobalAligner()
    
    # 对每个图像对运行 DUSt3R
    pairwise_poses = {}
    pointmaps = {}
    
    for i, img1_name in enumerate(image_files):
        for j, img2_name in enumerate(image_files):
            if i >= j:
                continue
            
            print(f"处理对 ({img1_name}, {img2_name})")
            
            # 这里应该加载实际图像并运行 DUSt3R
            # 现在演示合成数据
            
            # 创建合成 Pointmaps
            H, W = 64, 64
            points1 = np.random.randn(H, W, 3) * 0.5 + np.array([0, 0, 2])[None, None, :]
            conf1 = 0.7 + 0.3 * np.random.rand(H, W)
            
            points2 = np.random.randn(H, W, 3) * 0.5 + np.array([i, 0, 2])[None, None, :]
            conf2 = 0.7 + 0.3 * np.random.rand(H, W)
            
            pm1 = PointMap(points1, confidence=conf1)
            pm2 = PointMap(points2, confidence=conf2)
            
            # 相对位姿估计
            pts1 = pm1.points.reshape(-1, 3)
            pts2 = pm2.points.reshape(-1, 3)
            conf_flat = pm1.confidence.flatten()
            mask = conf_flat > conf_threshold
            
            if mask.sum() > 10:
                R, t = GlobalAligner.procrustes_align(pts1[mask], pts2[mask])
                pairwise_poses[(f'img_{i}', f'img_{j}')] = (R, t)
    
    # 添加帧和处理全局对齐
    for i, img_name in enumerate(image_files):
        # 创建 Pointmap
        H, W = 64, 64
        points = np.random.randn(H, W, 3) * 0.5 + np.array([i, 0, 2])[None, None, :]
        conf = 0.7 + 0.3 * np.random.rand(H, W)
        pm = PointMap(points, confidence=conf)
        
        aligner.add_frame(f'img_{i}', pm)
    
    # 全局对齐
    aligner.pairwise_to_global(pairwise_poses)
    
    print(f"\n处理完成！")
    print(f"  帧数: {len(aligner.points)}")
    print(f"  配对数: {len(pairwise_poses)}")
    
    return aligner

print("run_dust3r_on_scene 函数定义完成")

## 5. 后处理：点云清理和导出


In [ ]:
def export_point_cloud(aligner, output_file, conf_threshold=0.5):
    """
    将全局点云导出为 PLY 格式。
    
    Args:
        aligner: GlobalAligner 对象
        output_file: 输出文件路径 (.ply)
        conf_threshold: 置信度阈值
    """
    points_list = []
    
    # 收集所有帧的高置信度点
    for frame_id, pm in aligner.points.items():
        pts, colors = pm.to_pointcloud(conf_threshold)
        points_list.append(pts)
    
    if points_list:
        all_points = np.vstack(points_list)
        
        # 简单的 PLY 导出
        header = f'''ply
format ascii 1.0
element vertex {len(all_points)}
property float x
property float y
property float z
end_header
'''
        with open(output_file, 'w') as f:
            f.write(header)
            for pt in all_points:
                f.write(f"{pt[0]:.6f} {pt[1]:.6f} {pt[2]:.6f}\n")
        
        print(f"导出 {len(all_points)} 个点到 {output_file}")
    else:
        print("没有足够的置信度高的点")

print("export_point_cloud 函数定义完成")

## 6. 完整工作流示例


In [ ]:
# 完整工作流
print("完整 DUSt3R 工作流：")
print()
print("1. 数据准备")
print("   - 收集图像 (jpg/png)")
print("   - 组织为 images/ 目录")
print()
print("2. 运行 DUSt3R")
print("   - 对每个图像对: 前向传播")
print("   - 提取 Pointmaps + Confidence")
print()
print("3. 全局对齐")
print("   - 使用 Procrustes 估计位姿")
print("   - 全局位姿优化")
print()
print("4. 点云融合")
print("   - 合并所有帧的点")
print("   - 置信度过滤")
print()
print("5. 导出结果")
print("   - 保存为 PLY")
print("   - 可在 CloudCompare/Meshlab 中查看")

## 7. 故障排除

| 问题 | 原因 | 解决方案 |
|------|------|----------|
| 低置信度点 | 无纹理或过度运动 | 改进图像，增加视角覆盖 |
| 位姿估计失败 | 无法匹配点 | 检查图像是否足够相似 |
| 内存不足 | 图像太大 | 缩小图像分辨率 |
| 点云破碎 | 全局对齐差 | 增加图像对数量 |


## 8. Summary

**关键步骤：**
1. 准备图像数据（无需相机参数）
2. 对每个图像对运行 DUSt3R
3. 提取 Pointmaps 和置信度
4. Procrustes 位姿估计
5. 全局对齐
6. 点云融合和导出

**优势：**
- 无需相机标定
- 自动处理多视图
- 快速推理（<1秒每对）
- 可与下游任务集成（SfM, SLAM, 3DGS 等）

---

## Next Steps

- [Phase 4: Feed-forward 3DGS](../../phase4/00_phase4_overview.ipynb) — 学习如何用神经网络直接预测 Gaussians
- [Phase 5: Vision Geometry Geometry Transformer](../../phase5/00_phase5_overview.ipynb) — 多视图基础模型
